In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

**<font size="6" color="red">ch03.연관분석</font>**
- pip install apyori

# 1. 연관분석 개요
- 데이터들 사이의 자주 발생하는 속성을 찾고, 그 속성들 사이의 연관성이 어느정도 있는지를 분석
- 활용분야 : 상품진열, 사기보험적발, 신상품 카테고리 구성,...
```
조건 (left-hand side, item_base) : 오렌지 주스(x)=>결과(right-hand side, item_add) : 와인(y)

연관분석 지표
1. 지지도(support) : 전체 데이터 중, 조건과 결과 항목들이 포함된 거래 비율
    (함께 얼마나 자주 나타나는지) (x,y)의 항목수 / 전체 데이터수 = 0.2

2. 신뢰도(confidence) : 조건(x)이 발생했을 때, 결과가 동시에 일어날 확률
    (조건이 오면 얼마나 자주 결과가 오는지) (x=>y)의 항목수 / x가 나오는 항목수 = 0.5 

3. 향상도(lift) : 우연히 발생할 규칙은 아니었는지 확인
    
    1미만 : 독립적으로 나오는것보다 함께 나타날 가능성이 낮다.
    
    1    : 서로 독립적. 아무 연관성이 없다.
    
    1초과 : 양의 상관관계 (같이 잘나온다)
    
    (x=>y)의 지지도 / x의 지지도*y의 지지도 = 0.2 / (0.4*0.6) = 0.2/0.24 = 0.83333
```


# 2. 연관분석 구현

In [24]:
import csv
with open('data/cf_basket.csv', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [25]:
from apyori import apriori
rules = apriori(transaction,
              min_support=0.15,
              min_confidence=0.1,
              min_lift=1.001)
rules = list(rules)
len(rules)

6

In [26]:
rule = rules[5]
rule

RelationRecord(items=frozenset({'소주', '콜라', '와인'}), support=0.2, ordered_statistics=[OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'소주', '와인'}), confidence=0.25, lift=1.25), OrderedStatistic(items_base=frozenset({'소주', '와인'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25)])

In [28]:
# support = rule[1]
# ordered_st = rule[2]
# for item in ordered_st:
#     lhs = item[0]
#     lhs = ','.join([x for x in lhs])
#     rhs = [1]
#     rhs = ','.join([x for x in rhs])
#     confidence = item[2]
#     lift = item[3]
#     print(f"{lhs}=>{rhs} \t {support} \t {confidence}\t{lift}")

support = rule[1]
ordered_st = rule[2]
for item in ordered_st:
    # print(item)
    lhs = item[0]
    lhs = ','.join([x for x in lhs])
    rhs = item[1]
    rhs = ','.join([x for x in rhs])
    confidence = item[2]
    lift = item[3]
    print(f"{lhs}=>{rhs} \t {support} \t {confidence} \t {lift}")

콜라=>소주,와인 	 0.2 	 0.25 	 1.25
소주,와인=>콜라 	 0.2 	 1.0 	 1.25


In [29]:
# for rule in rules:
#     support = rule[1]
#     ordered_st = rule[2]
#     for item in ordered_st:        
#         lhs = item[0]
#         lhs = ','.join([x for x in lhs])
#         rhs = [1]
#         rhs = ','.join([x for x in rhs])
#         confidence = item[2]
#         lift = item[3]
# #         print(f"{lhs}=>{rhs} \t {support} \t {confidence}\t{lift}")
#         rules_lst.append({'lhs' : lhs,
#                          'rhs' : rhs,
#                          'support' : support,
#                          'confidence' : round(confidence,2),
#                           'lift' : round(lift,2)
#                          })
# import pandas as pd
# pd.DataFrame(rules_lst)

rules_lst = [] # 규칙을 저장할 dict list
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
        # print(item)
        lhs = item[0]
        lhs = ','.join([x for x in lhs])
        rhs = item[1]
        rhs = ','.join([x for x in rhs])
        confidence = item[2]
        lift = item[3]
        # print(f"{lhs}=>{rhs} \t {support} \t {round(confidence,2)} \t {round(lift,2)}")
#         rules_lst.append({'lhs':lhs,
#                          'rhs':rhs,
#                          'support':support,
#                          'confidence':round(confidence,2),
#                          'lift':round(lift,2)})
        rules_lst.append([lhs, rhs, support, round(confidence, 2), round(lift, 2)])
import pandas as pd
pd.DataFrame(rules_lst, columns=['lhs','rhs','support','confidence', 'lift'])

,lhs,rhs,support,confidence,lift
0,맥주,콜라,0.4,1.00,1.25
1,콜라,맥주,0.4,0.50,1.25
2,소주,콜라,0.6,1.00,1.25
3,콜라,소주,0.6,0.75,1.25
4,콜라,"맥주,소주",0.2,0.25,1.25
5,"맥주,소주",콜라,0.2,1.00,1.25
6,맥주,"콜라,와인",0.2,0.50,1.25
7,콜라,"맥주,와인",0.2,0.25,1.25
8,"맥주,와인",콜라,0.2,1.00,1.25
9,"콜라,와인",맥주,0.2,0.50,1.25


In [16]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [17]:
import os
import requests
import json
import pandas as pd
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

url = f"https://openapi.naver.com/v1/search/news.json" # JSON 결과
params = {'query':'스포츠', 'display':20 ,'sort':'date'}
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items'] # response를 json형태로 변환한것 중 'items'

items
# 제목(title)+' '+요약(description) 텍스만 추출하여 list
news_texts=[]
for item in items:
    title = item.get('title').replace('<b>',' ').replace('</b>',' ')
    description = item.get('description').replace('<b>',' ').replace('</b>',' ')
    news_texts.append(title+' '+description)
news_texts[:3],len(news_texts
                  )

(['금산군 2027년 제12회 국제오픈 태권도대회 유치 업무협약 체결 군은 이번 유치를 기점으로 국내외  스포츠  관광객을 적극적으로 유입해 지역 소상공인 매출 증대와 골목상권 활성화를 도모하고 세계 속에 글로벌  스포츠  건강 도시 금산의 이미지를 확고히 각인시킬 방침이다. 문정우... ',
  "[FIBA WC] '이탈리아이 진땀승' 스튜어트, &quot;그래도 자랑스럽다&quot;라고 말한... 팀의 베테랑인 브리애나 스튜어트는 경기 후 NBC 스포츠 와의 인터뷰를 통해 아쉬움을 전했다. 스튜어트는 &quot;이탈리아가 우리 경기를 상당히 엉망으로 만들었고, 득점하기 어렵게 만들었다. 그러면서 우리는 정말 힘들게... ",
  '전북 장애학생 20명, 전국 장애학생 e 스포츠 대회 출전 전북 지역 장애학생들이 내일(8일)부터 이틀간 장애학생들의 e 스포츠 대회인 전국 장애학생 e페스티벌에서 기량을 펼치고 올 예정입니다. 이번 대회에는 도내 예선전에 참가한 522명 중에서 최종 선발된 20명이 e 스포츠 대... '],
 18)

In [32]:
# 명사추출
stopwords = {'스포츠','기자'}
from konlpy.tag import Hannanum, Kkma, Komoran,Okt
from mecab import MeCab
analyzer = MeCab()
news = []
for article in news_texts:
    noun_list = analyzer.nouns(article)
    noun_list = [word for word,tag in analyzer.pos(article) if tag in('NNG','NNP')]
    news.append(noun_list)
print(news[:3])

[['금산군', '국제', '오픈', '태권', '대회', '유치', '업무', '협약', '체결', '군', '이번', '유치', '기점', '국내외', '스포츠', '관광객', '적극', '유입', '지역', '상공', '매출', '증대', '골목', '상권', '활성', '도모', '세계', '속', '글로벌', '스포츠', '건강', '도시', '금산', '이미지', '각인', '방침', '문정'], ['이탈리아', '진땀', '승', '스튜어트', '자랑', '말', '팀', '베테랑', '브리', '애나', '스튜어트', '경기', '후', '스포츠', '와의', '인터뷰', '아쉬움', '스튜어트', '이탈리아', '경기', '엉망', '득점'], ['전북', '장애', '학생', '전국', '장애', '학생', '스포츠', '대회', '출전', '전북', '지역', '장애', '학생', '내일', '이틀', '장애', '학생', '스포츠', '대회', '전국', '장애', '학생', '페스티벌', '기량', '예정', '이번', '대회', '도내', '예선전', '참가', '최종', '선발', '스포츠']]


In [33]:
rules = apriori(news,
               min_support=0.15,
               min_confidence=0.1,
               min_lift=1.00001)
rules = list(rules)
len(rules)

4

In [34]:
rules_lst = [] # 규칙을 저장할 dict list
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
        # print(item)
        lhs = item[0]
        lhs = ','.join([x for x in lhs])
        rhs = item[1]
        rhs = ','.join([x for x in rhs])
        confidence = item[2]
        lift = item[3]
        # print(f"{lhs}=>{rhs} \t {support} \t {round(confidence,2)} \t {round(lift,2)}")
#         rules_lst.append({'lhs':lhs,
#                          'rhs':rhs,
#                          'support':support,
#                          'confidence':round(confidence,2),
#                          'lift':round(lift,2)})
        rules_lst.append([lhs, rhs, support, round(confidence, 2), round(lift, 2)])
import pandas as pd
df = pd.DataFrame(rules_lst, columns=['lhs','rhs','support','confidence', 'lift'])


In [35]:
df.sort_values(by=['lift','confidence', 'support'], ascending=False, inplace=True)
df # 지지도는 신뢰성 체크용

,lhs,rhs,support,confidence,lift
3,지역,이번,0.166667,0.75,2.25
9,지역,"스포츠,이번",0.166667,0.75,2.25
11,"스포츠,지역",이번,0.166667,0.75,2.25
2,이번,지역,0.166667,0.50,2.25
8,이번,"스포츠,지역",0.166667,0.50,2.25
10,"스포츠,이번",지역,0.166667,0.50,2.25
0,대회,이번,0.166667,0.60,1.80
4,대회,"스포츠,이번",0.166667,0.60,1.80
6,"스포츠,대회",이번,0.166667,0.60,1.80
1,이번,대회,0.166667,0.50,1.80
